# Rag Pipeline - Data Ingestion to Vector DB pipline

## 1. Data Ingestion

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import TextSplitter, RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\abirf\AppData\Local\Temp\ipykernel_18088\1206334144.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [2]:
### Read all the pdfs inside the dictionary 

def pdf_reader(pdf_directory):
    """Process all the PDF files from directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all the pdf files in a list 
    pdf_list = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_list)} PDF(s) to process")

    for pdf_file in pdf_list:
        print(f"\nProcessing : {pdf_file.name}")
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()

        # adding some more information to metadata
        for doc in documents:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'

        all_documents.extend(documents)
        print(f"Loaded {len(documents)} pages")

    print(f"\nTotal pages {len(all_documents)}")
    return all_documents
# Process all PDFs in the data directory
all_pdf_document = pdf_reader("../data/pdf_file")


Found 2 PDF(s) to process

Processing : Cover_letter_Md_Abir__Data Science@SLB.pdf
Loaded 2 pages

Processing : CV_Md_Abir__Schibsted.pdf
Loaded 1 pages

Total pages 3


all_pdf_document

## 2. Chunking

In [3]:
def chunk_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        #length_funtion = len,
        separators= ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

In [4]:
chunks = chunk_documents(all_pdf_document)

Split 3 documents into 12 chunks


In [5]:
chunks

[Document(metadata={'producer': 'Skia/PDF m157 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Cover_letter_Md_Abir<>Data Science@SLB', 'source': '..\\data\\pdf_file\\Cover_letter_Md_Abir__Data Science@SLB.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Cover_letter_Md_Abir__Data Science@SLB.pdf', 'file_type': 'pdf'}, page_content='Md\n \nAbir\n \nHossain\n \nBirmingham,\n \nUK\n \n|\n \nabirfarabi915@gmail.com\n \n|\n \n+44\n \n7507034017\n \n \nDate:\n \n12/10/2026\n \n \nDear\n \nSLB\n \nTeam,\n \nI\n \nam\n \nwriting\n \nfor\n \nthe\n \nData\n \nScience\n \n-\n \nIndustrial\n \nData\n \n&\n \nAnalytics\n \nrole\n \nat\n \nSLB.\n \nI\n \nrecently\n \ncompleted\n \nmy\n \nMSc\n \nin\n \nArtificial\n \nIntelligence\n \nand\n \nMachine\n \nLearning\n \nat\n \nthe\n \nUniversity\n \nof\n \nBirmingham\n \nwith\n \na\n \nresearch\n \nbackground\n \nin\n \nMachine\n \nLearning\n \nwith\n \nmedical\n \nsensor\n \ndata.\n \nPrior\n \nto\n \nthis

## 3. Embeddings 

### Importing Libraries

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import tqdm


c:\Users\abirf\OneDrive\Desktop\AI Course\RAG\myrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()  


        

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading the embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
        except Exception as e:
            print(f"Error loading the model: {self.model_name}")
            raise



    def generate_embeddings(self, texts: List[str]) -> np.array:
        """
        Generate embeddings for a list of texts

        Args: 
            texts: List of text string to embed

        Return:
            np.array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ..#..#..#")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Embeddings completed\nThe generated dimension is: {embeddings.shape}")

        return embeddings

In [8]:
### Initialize the embedding manager 

embedding_manager = EmbeddingManager()

Loading the embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2890.73it/s]


## 4. Vector Store

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_document", persist_directory : str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args: 
            collection_name : Name of the ChromaDB collection
            persist_directory : Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistant ChromaDB client
            os.makedirs(self.persist_directory, exist_ok= True)
            self.client = chromadb.PersistentClient(path= self.persist_directory)

            ## Get or create collection 
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG",
                            "hnsw:space" : "cosine"}
            )

        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corrosponding embeddings for the documents
        
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match with number of embeddings")
        print(f"Adding {len(documents)} documents to vector store")

        # Prepare data for chroma db
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = 1
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas= metadatas,
                documents=documents_text
            )
            print(f"Successfully add {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [10]:
vectorstore = VectorStore()
vectorstore

In [11]:
vectorstore.client.delete_collection("pdf_document")   # wipes the old L2 collection


### Converting text to embeddings

In [12]:
texts = [doc.page_content for doc in chunks]

In [13]:
## Generate the embeddings now 

embeddigns = embedding_manager.generate_embeddings(texts)

Generating embeddings for 12 texts ..#..#..#


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.67it/s]

Embeddings completed
The generated dimension is: (12, 384)


In [14]:
vectorstore = VectorStore()                      # creates a fresh collection + new handle
print(vectorstore.collection.metadata)           # should show 'hnsw:space': 'cosine'
vectorstore.add_documents(chunks, embeddigns)    # should say: Total documents in collection: 22


{'hnsw:space': 'cosine', 'description': 'PDF document embeddings for RAG'}
Adding 12 documents to vector store
Successfully add 12 documents to vector store
Total documents in collection: 12


## 5. Retriever Pipeline From VectorStore

In [15]:
class RAGRetriever:
    """Handles query based retrieval from vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever 

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query from the user
            top_k: Number of top result to return 
            score_threshold: Minimum similarity score threshold

        Return: 
            List of dictionaries containing retrieved documents and metadatas
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score Threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results= top_k
            )

            #Processing the results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                'id': doc_id,
                                'content' : document,
                                'metadata' : metadata,
                                'similarity_score' : similarity_score,
                                'distance' : distance,
                                'rank' : i+1
                            }
                        )
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [16]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)   # rebuild so it uses the new store

## 6. Integration VectorDB context pipeline with LLM output

In [30]:
## Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(
    groq_api_key = groq_api_key,
    model = "openai/gpt-oss-20b",
    temperature= 0.1,
    max_tokens= 1024
)

In [31]:
## Simple RAG funtion : retrieve context + generate response 

def rag_simple(query, retriever, llm, top_k = 3):
    ## retrieving the context
    results = retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent contect found"
    ## generate the answer using GROQ
    prompt = f"""Use the following context to answer the question concisely
        Context: {context}
    
        Question: {query}

        Answer: """

    response = llm.invoke([prompt.format(
        context = context,
        query = query
    )])

    return response.content


In [42]:
answer = rag_simple("What kind of jobs he is looking for?", rag_retriever, llm)

Retrieving documents for query: 'What kind of jobs he is looking for?'
Top k: 3, Score Threshold: 0.0
Generating embeddings for 1 texts ..#..#..#


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.44it/s]

Embeddings completed
The generated dimension is: (1, 384)
Retrieved 3 documents (after filtering)


In [37]:
answer

'He is targeting data‑science and AI/ML roles—specifically positions that blend industrial data analytics, machine‑learning engineering, data‑pipeline development, and generative‑AI application (e.g., Data Scientist/ML Engineer in an industrial analytics setting).'

## 7. Enhanced RAG pipeline Features

In [60]:
def rag_adanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score and optionally full context.
    """
    results =retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results:
        return {'answer' : 'No relevant context found',
                'source' : [],
                'confidence' : 0.0,
                'context' : ""}
    # Prepare context and source 
    context = "\n\n".join([doc['content'] for doc in results])
    source = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:120]+'...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])


    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext: {context}.\nQuery: {query}.\nAnswer:"""
    
    response = llm.invoke([prompt.format(
            context = context,
            query = query
        )])

   

    output = {
        'answer' : response.content,
        'sources': source,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

results = rag_adanced("Who is Abir?", rag_retriever, llm, top_k=3, min_score= 0.1, return_context= True)
print("Answer:", results['answer'])
print("Sources:", results['sources'])
print("Confidence:", results['confidence'])

Retrieving documents for query: 'Who is Abir?'
Top k: 3, Score Threshold: 0.1
Generating embeddings for 1 texts ..#..#..#


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.68it/s]

Embeddings completed
The generated dimension is: (1, 384)
Retrieved 2 documents (after filtering)


Answer: Abir Hossain is a data‑science professional based in Birmingham, UK, holding an MSc in Artificial Intelligence and Machine Learning from the University of Birmingham. He has hands‑on experience building machine‑learning and AI models—including computer vision, time‑series forecasting, and LLMs—and worked as a Data Science Intern at Aziz & Brothers, where he analysed large customer datasets and developed forecasting models to drive business decisions.
Sources: [{'source': 'CV_Md_Abir__Schibsted.pdf', 'page': 0, 'score': 0.2907874584197998, 'preview': 'Md Abir Hossain\n+44 7507034017 | abirfarabi915@gmail.com | linkedin.com/in/abir-farabi | Github | Birmingham, UK\nSummary...'}, {'source': 'Cover_letter_Md_Abir__Data Science@SLB.pdf', 'page': 0, 'score': 0.15994149446487427, 'preview': 'Md\n \nAbir\n \nHossain\n \nBirmingham,\n \nUK\n \n|\n \nabirfarabi915@gmail.com\n \n|\n \n+44\n \n7507034017\n \n \nDate:\n \n12/10/2026\n \n \nD...'}]
Confidence: 0.2907874584197998


In [59]:
answerad = rag_adanced("What about slotfinder?", rag_retriever, llm, top_k=3, min_score= 0.1, return_context= True)
print(answerad)


Retrieving documents for query: 'What about slotfinder?'
Top k: 3, Score Threshold: 0.1
Generating embeddings for 1 texts ..#..#..#


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.40it/s]


Embeddings completed
The generated dimension is: (1, 384)
Retrieved 3 documents (after filtering)
{'answer': 'SlotFinder was a project where I tackled forecasting on messy, real‑world time series that lacked a clean physical model. It taught me how to extract meaningful insights from operational data generated by real physical systems.', 'sources': [{'source': 'Cover_letter_Md_Abir__Data Science@SLB.pdf', 'page': 0, 'score': 0.36973321437835693, 'preview': 'me\n \nunderstand\n \nwhich\n \nparts\n \nof\n \nthe\n \nmodel\n \nactually\n \nmattered.\n \nIt\n \nwas\n \nmy\n \nfirst\n \nreal\n \nexperience\n \n...'}, {'source': 'Cover_letter_Md_Abir__Data Science@SLB.pdf', 'page': 0, 'score': 0.31729209423065186, 'preview': 'with\n \ntime\n \nseries\n \nforecasting\n \nin\n \nSlotFinder\n \nand\n \nsensor\n \ndata\n \nmodelling\n \nin\n \nmy\n \ndissertation\n \nshowed...'}, {'source': 'Cover_letter_Md_Abir__Data Science@SLB.pdf', 'page': 1, 'score': 0.24141573905944824, 'preview': 'test\n \